# AI-Powered Contract Review and Risk Analysis Platform — Live Demo Notebook
**Use this during your capstone presentation.**

Runs end-to-end on a single sample contract in < 2 minutes.
No dataset download needed — uses a synthetic contract snippet.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
print('Ready. API key:', 'SET' if os.getenv('OPENAI_API_KEY') else 'MISSING')

In [ ]:
# Sample contract text (use your own PDF in the Streamlit app)
SAMPLE_CONTRACT = """
SERVICE AGREEMENT

1. LIMITATION OF LIABILITY
In no event shall either party be liable for any indirect, incidental, special,
consequential, or punitive damages, regardless of the cause of action, even if advised
of the possibility of such damages. The maximum aggregate liability of either party
shall not exceed the total fees paid by Client in the three (3) months immediately
preceding the claim.

2. TERMINATION
Either party may terminate this Agreement upon thirty (30) days prior written notice.
The Company may terminate this Agreement immediately upon material breach by the Client
that remains uncured for ten (10) days following written notice.

3. PAYMENT TERMS
Client shall pay all invoices within forty-five (45) days of receipt. Overdue amounts
shall accrue interest at 1.5% per month. All fees are non-refundable.

4. CONFIDENTIALITY
Each party agrees to maintain the confidentiality of the other party's Confidential
Information and not to disclose it to any third party without prior written consent.
This obligation survives termination for a period of five (5) years.

5. INDEMNIFICATION
Client shall indemnify, defend, and hold harmless the Company from any claims, damages,
losses, and expenses (including reasonable attorneys' fees) arising from Client's use
of the services or breach of this Agreement.

6. GOVERNING LAW
This Agreement shall be governed by and construed in accordance with the laws of the
State of Delaware, without regard to conflict of laws principles.
"""

print('Contract loaded:', len(SAMPLE_CONTRACT.split()), 'words')

### Step 1: Chunk the contract

In [ ]:
from modules.doc_processor import chunk_pages

pages = [{'page_num': 1, 'text': SAMPLE_CONTRACT}]
chunks = chunk_pages(pages, source_file='demo_contract.txt', chunk_size=120, overlap=20)

print(f'{len(chunks)} chunks extracted')
for i, c in enumerate(chunks):
    print(f'  Chunk {i+1}: {c.text[:70]}...')

### Step 2: Topic modelling — LLM zero-shot classification

In [ ]:
from modules.topic_modeler import classify_chunks_llm

chunk_texts = [c.text for c in chunks]
print('Classifying chunks with LLM...')
topic_results = classify_chunks_llm(chunk_texts)

print('\nChunk → Clause type mapping:')
for text, result in zip(chunk_texts, topic_results):
    print(f'  {result["clause_type"]:35s} | risk={result["risk_level"]:6s} | conf={result["confidence"]:.2f}')
    print(f'    {text[:70]}...')

### Step 3: Embedding-based clause matching

In [ ]:
from modules.clause_matcher import match_chunks, summarize_risk_profile
import pandas as pd

print('Running cosine similarity matching...')
match_results = match_chunks(chunk_texts, threshold=0.35)

df = pd.DataFrame([{
    'text': r['text'][:55] + '...',
    'matched_clause': r['matched_clause'],
    'match_score': r['match_score'],
    'risk_level': r['risk_level'],
    'risk_score': r['risk_score'],
} for r in match_results])

print(df.to_string(index=False))
print('\nRisk profile summary:')
print(summarize_risk_profile(match_results))

### Step 4: Index into ChromaDB + RAG retrieval

In [ ]:
from modules.embedder import index_chunks, semantic_search

print('Indexing into ChromaDB...')
index_chunks(chunks, topic_labels=match_results)

query = 'What is the maximum liability cap?'
print(f'\nQuery: "{query}"')
results = semantic_search(query, top_k=3)

for i, r in enumerate(results):
    print(f'\n  Result {i+1} (similarity={r["similarity"]})')
    print(f'  Clause: {r["metadata"].get("clause_type", "?")} | Risk: {r["metadata"].get("risk_level", "?")}') 
    print(f'  Text: {r["text"][:120]}...')

### Step 5: Full LLM clause extraction (structured output)

In [ ]:
from modules.llm_engine import extract_clauses

print('Running full contract analysis...')
analysis = extract_clauses(SAMPLE_CONTRACT, contract_name='demo_contract.txt')

print(f'\nContract: {analysis.contract_name}')
print(f'Overall risk: {analysis.overall_risk_level.upper()}')
print(f'Summary: {analysis.executive_summary}')

print(f'\nExtracted {len(analysis.clauses)} clauses:')
for clause in analysis.clauses:
    print(f'  [{clause.risk_level.upper():6s}] {clause.clause_type:35s} | {clause.summary[:60]}')

if analysis.red_flags:
    print(f'\nRed flags ({len(analysis.red_flags)}):')
    for flag in analysis.red_flags:
        print(f'  ! {flag}')

if analysis.missing_clauses:
    print(f'\nMissing standard clauses:')
    for c in analysis.missing_clauses:
        print(f'  - {c}')

### Step 6: Chat Q&A with RAG

In [ ]:
from modules.rag_pipeline import rag_retrieve_and_assemble
from modules.llm_engine import answer_question, answer_for_voice

questions = [
    'What is the liability cap in this contract?',
    'Can the company terminate the agreement immediately?',
    'How long does the confidentiality obligation last after termination?',
]

chat_history = []
for question in questions:
    context, chunks_used = rag_retrieve_and_assemble(question, top_k=4)
    answer = answer_question(question, context, chat_history=chat_history)

    print(f'Q: {question}')
    print(f'A: {answer.answer}')
    print(f'   Sources: {answer.sources} | Confidence: {answer.confidence:.2f}')
    print(f'   Voice output: {answer_for_voice(answer)}')
    print()

    chat_history.append({'role': 'user', 'content': question})
    chat_history.append({'role': 'assistant', 'content': answer.answer})

### Step 7: Voice transcription test (Whisper)

In [ ]:
# This cell demos TTS — generates audio of the last answer
from modules.voice_handler import text_to_speech
import IPython.display as ipd

last_answer = chat_history[-1]['content']
print(f'Generating TTS for: "{last_answer[:80]}..."')

audio_bytes = text_to_speech(last_answer, voice='nova')

# Play inline in notebook
ipd.display(ipd.Audio(audio_bytes, autoplay=False))
print(f'Audio generated: {len(audio_bytes):,} bytes')

---
**Demo complete.**

For the full evaluation on CUAD (50 contracts), run `capstone_evaluation.ipynb`.

For the interactive app: `streamlit run app/main_app.py`